# AGender-MTL - Step F: figures and tables (CPU)

Draws every figure and table of the article from the files Steps 1-3 saved. Nothing is typed in. The only
training is one repeat (seed 42) of the network for the 6- and 10-class schemes, because confusion
matrices, ROC curves and permutation importance need test predictions; it is the same model as in Step 1.

**Kaggle settings:** Accelerator = **None**, Internet = Off is fine.
**Inputs to add:** `agender-mtl-code` and the outputs of Steps 0, 1, 2 and 3 (as datasets, or as notebooks
with Add Input -> Notebooks). **Expected run time:** about 5 minutes.

In [ ]:
import os
import sys
from pathlib import Path

# The code is attached as the Kaggle dataset `agender-mtl-code`. Find the `agender_mtl` package
# wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('benchmark.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The agender-mtl-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> agender-mtl-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if package.parent.name == 'agender_mtl' and (package.parent / '__init__.py').exists():
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder; link the files under the name `agender_mtl`.
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'agender_mtl').exists():
        os.symlink(package.parent, code_root / 'agender_mtl')

sys.path.insert(0, str(code_root))
import agender_mtl
print('Code loaded from', Path(agender_mtl.__file__).parent)

In [ ]:
OUTPUT = Path('/kaggle/working/agender_mtl_outputs/stepF')
INPUT = Path('/kaggle/input')

def folder_with(name):
    found = find_file(name, INPUT)
    return found.parent if found else None

step1_dir = folder_with('scores_6class.csv')
step2_dir = find_file('step2_summary.json', INPUT).parent if find_file('step2_summary.json', INPUT) else None
step3_dir = folder_with('qubo_choice.json')
lat_path = find_file('lat_s1.zip', INPUT) or find_file('age-gender-pred-dataset.csv', INPUT)
for name, where in (('LAT', lat_path), ('Step 1', step1_dir), ('Step 2', step2_dir), ('Step 3', step3_dir)):
    print(name, where)
if lat_path is None or step1_dir is None:
    raise RuntimeError('Attach the outputs of Step 0 and Step 1 (Steps 2 and 3 are optional).')

In [ ]:
import json

from agender_mtl import data, figures

raw = data.read_lat(lat_path)
numbers = figures.run(raw, step1_dir, step2_dir or OUTPUT / 'none', step3_dir or OUTPUT / 'none', OUTPUT)
print(json.dumps({k: v for k, v in numbers.items() if k != 'schemes'}, indent=1, default=str)[:3000])
print(sorted(p.name for p in (OUTPUT / 'figures').glob('*.pdf')))
print(sorted(p.name for p in (OUTPUT / 'tables').glob('*.tex')))